# 예제 05. 결과 분석과 모델 비교
빅데이터프로그래밍 · 12주차

## 목표
- 직접 쓴 문장으로 예측을 확인한다
- 틀린 예측의 원인을 추정한다
- 여러 모델 구조를 비교한다

과제와 같은 형식입니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import re, random, os, urllib.request, tarfile
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

torch.manual_seed(42); random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 데이터 (예제 04와 같음)


In [ ]:
URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
if not os.path.exists("aclImdb"):
    urllib.request.urlretrieve(URL, "aclImdb.tar.gz")
    with tarfile.open("aclImdb.tar.gz") as f:
        f.extractall(".")

PAD, UNK = 0, 1
MAX_LEN, MAX_VOCAB = 200, 10000

def tokenize(s):
    s = s.lower().replace("<br />", " ")
    return re.sub(r"[^a-z0-9'\s]", " ", s).split()

def load_split(split, n=2500):
    data = []
    for label, name in [(1, "pos"), (0, "neg")]:
        d = f"aclImdb/{split}/{name}"
        for fn in sorted(os.listdir(d))[:n]:
            with open(os.path.join(d, fn), encoding="utf-8") as f:
                data.append((f.read(), label))
    random.shuffle(data)
    return data

train_data, test_data = load_split("train"), load_split("test", 1250)
train_tokens = [tokenize(t) for t, _ in train_data]
test_tokens  = [tokenize(t) for t, _ in test_data]

counter = Counter(w for t in train_tokens for w in t)
vocab = {"<PAD>": PAD, "<UNK>": UNK}
for w, _ in counter.most_common(MAX_VOCAB - 2):
    vocab[w] = len(vocab)


class ReviewDataset(Dataset):
    def __init__(self, tl, labels, max_len=MAX_LEN):
        seqs = []
        for t in tl:
            ids = [vocab.get(w, UNK) for w in t][:max_len]
            seqs.append(ids + [PAD] * (max_len - len(ids)))
        self.x = torch.tensor(seqs); self.y = torch.tensor(labels)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]


train_loader = DataLoader(ReviewDataset(train_tokens, [l for _, l in train_data]),
                          batch_size=64, shuffle=True)
test_loader  = DataLoader(ReviewDataset(test_tokens, [l for _, l in test_data]),
                          batch_size=128, shuffle=False)
LABELS = ["부정", "긍정"]
print("준비 완료 · 사전", len(vocab))


## 2. 세 가지 모델 구조


In [ ]:
class MeanPool(nn.Module):
    """임베딩 평균만 쓰는 가장 단순한 모델 — 순서를 무시합니다"""
    def __init__(self, V, emb=64):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=PAD)
        self.fc = nn.Linear(emb, 2)
    def forward(self, x):
        mask = (x != PAD).unsqueeze(-1).float()
        e = self.emb(x) * mask
        avg = e.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.fc(avg)


class LSTMModel(nn.Module):
    def __init__(self, V, emb=64, hidden=64, bidirectional=False):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=PAD)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True, bidirectional=bidirectional)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), 2)
    def forward(self, x):
        out, _ = self.lstm(self.emb(x))
        return self.fc(self.drop(out[:, -1, :]))


for name, m in [("평균 풀링", MeanPool(len(vocab))),
                ("LSTM", LSTMModel(len(vocab))),
                ("양방향 LSTM", LSTMModel(len(vocab), bidirectional=True))]:
    print(f"{name:12s} 파라미터 {sum(p.numel() for p in m.parameters()):,}개")


## 3. 학습과 비교


In [ ]:
loss_fn = nn.CrossEntropyLoss()

def evaluate(model):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in test_loader:
            preds.append(model(x.to(device)).argmax(dim=1).cpu()); trues.append(y)
    p, t = torch.cat(preds), torch.cat(trues)
    return (p == t).float().mean().item(), p, t


def run(name, model, epochs=6, lr=1e-3):
    torch.manual_seed(42)
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    print(f"\n{name}")
    start = time.time()
    for epoch in range(1, epochs+1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        acc, _, _ = evaluate(model)
        print(f"  epoch {epoch}  검증 {acc:.4f}")
    acc, p, t = evaluate(model)
    return {"model": model, "acc": acc, "pred": p, "true": t,
            "time": time.time()-start,
            "params": sum(q.numel() for q in model.parameters())}


results = {}
results["평균 풀링"]  = run("평균 풀링", MeanPool(len(vocab)))
results["LSTM"]       = run("LSTM", LSTMModel(len(vocab)))
results["양방향 LSTM"] = run("양방향 LSTM", LSTMModel(len(vocab), bidirectional=True))


In [ ]:
table = pd.DataFrame([{"모델": k, "파라미터": f"{r['params']:,}",
                       "검증 정확도": round(r["acc"], 4),
                       "학습 시간(초)": round(r["time"], 1)}
                      for k, r in results.items()])
print(table.to_string(index=False))

best = table.loc[table["검증 정확도"].idxmax(), "모델"]
print(f"\n가장 좋은 모델: {best}")


평균 풀링은 순서를 완전히 무시하는데도 꽤 잘합니다 — 감성 분류에서는 어떤 단어가 나왔는지가 큰 힌트이기 때문입니다.


## 4. 직접 쓴 문장 10개로 예측


In [ ]:
model = results[best]["model"]

def predict(text, model=model):
    toks = tokenize(text)
    ids = [vocab.get(w, UNK) for w in toks][:MAX_LEN]
    n_unk = sum(1 for i in ids if i == UNK)
    ids += [PAD] * (MAX_LEN - len(ids))
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(torch.tensor([ids]).to(device)), dim=1).cpu().squeeze()
    return {"예측": LABELS[p.argmax()], "확률": round(p.max().item(), 3),
            "단어 수": len(toks), "UNK": n_unk}


my_sentences = [
    ("This is one of the best films I have ever seen", 1),
    ("Absolutely awful, I walked out halfway through", 0),
    ("The acting was great but the story made no sense", 0),
    ("not bad", 1),
    ("I did not hate it", 1),
    ("A masterpiece of modern cinema", 1),
    ("Boring, predictable and far too long", 0),
    ("It was okay I guess", 0),
    ("The visuals were stunning even if the plot was thin", 1),
    ("Never again", 0),
]

rows = []
for text, truth in my_sentences:
    r = predict(text)
    r["문장"] = text[:45]
    r["정답"] = LABELS[truth]
    r["맞음"] = "O" if r["예측"] == LABELS[truth] else "X"
    rows.append(r)

df = pd.DataFrame(rows)[["문장", "정답", "예측", "확률", "단어 수", "UNK", "맞음"]]
print(df.to_string(index=False))
print(f"\n{(df['맞음'] == 'O').sum()} / {len(df)} 맞음")


## 5. 틀린 예측의 원인 추정

| 원인 | 확인 방법 |
| --- | --- |
| 부정 표현 | `not bad`, `did not hate` — 앞뒤 관계를 못 잡음 |
| 문장이 너무 짧음 | 단어 수 열을 보세요 |
| UNK가 많음 | UNK 열을 보세요 — 사전에 없는 단어가 많으면 판단 근거가 사라집니다 |
| 긍정과 부정이 섞임 | "acting was great but story made no sense" |
| 애매한 표현 | "okay I guess" — 사람도 애매합니다 |


In [ ]:
wrong = df[df["맞음"] == "X"]
if len(wrong):
    print("틀린 문장 분석:")
    for _, r in wrong.iterrows():
        reasons = []
        if r["단어 수"] < 5: reasons.append("문장이 짧음")
        if r["UNK"] > 0: reasons.append(f"UNK {r['UNK']}개")
        if r["확률"] < 0.7: reasons.append("모델도 확신 없음")
        if any(w in r["문장"].lower() for w in ["not ", "but ", "never"]):
            reasons.append("부정·역접 표현")
        print(f"\n  {r['문장']}")
        print(f"    정답 {r['정답']} / 예측 {r['예측']} ({r['확률']})")
        print(f"    추정 원인: {', '.join(reasons) if reasons else '불명'}")
else:
    print("전부 맞혔습니다")


## 6. UNK 비율과 정확도의 관계


In [ ]:
unk_rates, corrects = [], []
model.eval()
with torch.no_grad():
    for i, (t, label) in enumerate(test_data[:1000]):
        toks = tokenize(t)[:MAX_LEN]
        ids = [vocab.get(w, UNK) for w in toks]
        rate = sum(1 for x in ids if x == UNK) / max(len(ids), 1)
        ids += [PAD] * (MAX_LEN - len(ids))
        p = model(torch.tensor([ids]).to(device)).argmax().item()
        unk_rates.append(rate); corrects.append(p == label)

unk_rates = np.array(unk_rates); corrects = np.array(corrects)
rows = []
for lo, hi in [(0, .02), (.02, .05), (.05, .1), (.1, 1)]:
    mask = (unk_rates >= lo) & (unk_rates < hi)
    if mask.sum() > 10:
        rows.append({"UNK 비율": f"{lo:.0%}~{hi:.0%}", "개수": int(mask.sum()),
                     "정확도": round(corrects[mask].mean(), 4)})
pd.DataFrame(rows)


## 직접 해보기
1. 자기 문장 10개를 만들어 예측시키고 틀린 것의 원인을 적으세요.
2. 사전 크기를 3,000으로 줄이면 UNK 비율과 정확도가 어떻게 되나요?
3. 부정 표현("not good", "hardly worth it")을 여러 개 넣어 모델이 잡는지 확인하세요.


In [ ]:
# 여기에 작성하세요
